In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from urllib.parse import urlparse
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv
import time,os,json
from groq import Groq
from openai import OpenAI
import gradio as gr

In [ ]:
# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')    
MODEL = 'gpt-4o'
openai = OpenAI()

In [6]:
def scrape_forum(company_name):
    # --- Headless setup ---
    url = "https://forum.valuepickr.com/"
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")  # modern headless mode
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)

    wait = WebDriverWait(driver, 10)

    # Open search
    wait.until(EC.presence_of_element_located((By.ID, "search-button"))).click()

    # Enter query
    search_box = wait.until(EC.presence_of_element_located((By.ID, "icon-search-input")))
    search_box.send_keys(company_name)
    search_box.send_keys(Keys.RETURN)

    # Wait for initial results
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "li.item")))

    # Click "More..."
    try:
        more_button = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".search-menu__show-more a.search-link"))
        )
        more_button.click()
    except:
        print("No 'More...' button found")

    # Wait for full results page
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".fps-result")))

    results = driver.find_elements(By.CSS_SELECTOR, ".fps-result")

    data = []

    for item in results:
        try:
            url = item.find_element(By.CSS_SELECTOR, "a.search-link").get_attribute("href")
            title = item.find_element(By.CSS_SELECTOR, ".topic-title").text
            blurb = item.find_element(
                By.CSS_SELECTOR,
                ".blurb.container > span:not(.date)"
            ).text

            data.append({
            "title": str(title),
            "url": str(url),
            "blurb": str(blurb)
            })

        except Exception as e:
            print("Skipping:", e)

    driver.quit()

    return data

In [7]:
link_system_prompt = """
You are provided with a list of links found on a financial forum. Each link has a title, a URL, and a blurb
describing the content of the page it points to.here is an example of the data you receive:
[
    {
        "title": "Infosys Annual Report 2023",
        "url": "https://infosys.com/annual-report-2023",
        "blurb": "Review the latest annual report for Infosys, detailing their financial performance and strategic initiatives."
    }
]
You are able to decide which of the links would be most relevant to someone researching the company,
and you can also infer the type of discussion it is (e.g. an annual report, a news article, a blog post, etc.)
based on the title and blurb.
You should respond in same JSON format of only those links that you think are relevant
"""

In [8]:
def select_relevant_links(company_name):
    scraped_data = scrape_forum(company_name)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": json.dumps(scraped_data)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    #print(f"Found {len(links)} relevant links:")
    return links
    

In [9]:
def clean_url(url):
    parts = url.split("/")[-2]
    if parts.isdigit():
     return "/".join(url.split("/")[:-1])
    else:
     return url
  
def select_paginated_links(company_name):

    links = select_relevant_links(company_name)

    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=chrome_options)

    all_pagination_urls = []

    for item in links:
        link = item["url"]
        cleaned_url = clean_url(link)
        driver.get(cleaned_url)
        wait = WebDriverWait(driver, 20)

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        try:
            total_element = wait.until(
                EC.presence_of_element_located(
                    (By.CLASS_NAME, "timeline-replies")
                )
            )
            total = total_element.text.split('/')[-1].strip()
            final_link = f"{cleaned_url}/{total}"  # use cleaned_url as base
        except Exception as e:
            print("Error finding total element:", e)
            final_link = cleaned_url  # fallback to cleaned_url on failure

        all_pagination_urls.append(final_link)  # ✅ inside loop now

    driver.quit()

    return all_pagination_urls

In [10]:
def scrape_comments(company_name):
    pagination_links = select_paginated_links(company_name)
    
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=chrome_options)

    all_text = []

    for link in pagination_links:
        driver.get(link)
        driver.execute_script("window.scrollTo(0, 0);")
        
        try:
            post_stream = driver.find_element(By.CLASS_NAME, "post-stream")
            paragraphs = post_stream.find_elements(By.TAG_NAME, "p")
            
            link_text = []
            for p in paragraphs:
                text = p.text.strip()
                if text:
                    link_text.append(text)

            # truncate per link
            joined = "\n\n".join(link_text)
            truncated = " ".join(joined.split()[:4000])  # ✅ 5000 words per link
            all_text.append(truncated)

        except Exception as e:
            print(f"Error scraping {link}:", e)

    driver.quit()

    return "\n\n".join(all_text)  # concat all links at the end

In [11]:
Analyst_system_prompt = """
You are a highly skilled financial analyst with expertise in equity research, financial statement analysis, and industry evaluation.

You are given raw discussion excerpts ("discussion pointers") from financial forums about a specific company. These may include opinions, partial data, speculation, and noise.

Your task is to synthesize this information into a clear, structured, and insightful one-page investment brief that helps an investor make an informed decision.

Guidelines:
- Extract signal from noise. Ignore irrelevant or low-quality commentary.
- Prioritize insights related to:
  - Financial performance (revenue, margins, ROCE/ROE, cash flows)
  - Business model and unit economics
  - Competitive positioning and peer comparison
  - Industry trends and tailwinds/headwinds
  - Government policy/regulatory impact (if applicable)
  - Management quality and capital allocation
- Where claims are made in the discussion, assess their credibility. Do not blindly accept statements.
- If data is incomplete, infer cautiously and explicitly mention assumptions.
- Highlight both bullish and bearish perspectives.


Rules:
- Be concise but information-dense.
- Avoid generic textbook explanations.
- Do not hallucinate numbers; only use or infer from given text.
- Do not include disclaimers like "this is not financial advice".
- Write in a professional, neutral tone.
"""

In [12]:
def get_Financial_user_prompt(company_name):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of the raw discussion excerpts ("discussion pointers") from financial forums about this company.
These may include opinions, partial data, speculation, and noise.;

use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += scrape_comments(company_name)
    #user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
def create_report(company_name):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": Analyst_system_prompt},
            {"role": "user", "content": get_Financial_user_prompt(company_name)}
        ],
        stream=True
    )
    #result = response.choices[0].message.content
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [20]:
message_input = gr.Textbox(label="Company Name:", info="Enter a company name to get a detailed one pager from our FinanceGPT", lines=1)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=create_report,
    title="FinanceGPT: Your AI Financial Analyst", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Shri Piston and Rings ltd.",
        "Tata Motors Ltd.",
        ], 
    flagging_mode="never"
    )
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
